# AgentCore Integration — Runtime Sessions and Persistent Memory

In the previous notebooks, we built the Intake Orchestrator — a supervisor agent that retrieves documents from S3, runs a pre-processing graph (authenticate → extract → verify), and persists the claim record to DynamoDB. It works, but it has two production gaps:

1. **No session isolation.** The pipeline runs in the notebook kernel. In production, two concurrent claims would share the same Python process, the same global state, the same `auth_passed` flag. One claim’s authentication failure could bleed into another claim’s pipeline run.

2. **No memory across phases.** After the pipeline completes, the supervisor’s message history is gone. When the Adjudicator (Phase 2) needs the verification decision from Phase 1, it has no way to retrieve it without re-running the entire pre-processing graph.

This notebook solves both problems with **Amazon Bedrock AgentCore**:

| Problem | AgentCore Solution | What It Does |
|---|---|---|
| No session isolation | **AgentCore Runtime** | Each claim gets its own microVM with isolated compute, memory, and filesystem. Sessions are managed with unique IDs. |
| No cross-phase memory | **AgentCore Memory** | Conversation history (short-term) and extracted facts (long-term) persist across sessions. Downstream agents retrieve context without re-running upstream work. |

### What You’ll Build

| Part | What Happens |
|---|---|
| **Part A** | Run the orchestrator pipeline and expose the stateless problem (the "before") |
| **Part B** | Deploy the Intake Orchestrator to AgentCore Runtime with session isolation |
| **Part C** | Add AgentCore Memory (short-term) — capture the full pipeline conversation |
| **Part D** | Add AgentCore Memory (long-term) — extract facts for cross-session retrieval |
| **Part E** | The payoff: retrieve Phase 1 context from a new session (the "after") |
| **Part F** | Cleanup |

## Setup

We reuse the shared agent modules and AWS configuration from previous notebooks. Additionally, we install and import the AgentCore SDK for Runtime and Memory integration.

In [ ]:
!pip install -q --upgrade bedrock-agentcore strands-agents>=1.36.0 strands-agents-tools typing_extensions

In [ ]:
import boto3
import json
import sys
import os
import re
import time
import uuid
from pathlib import Path
from datetime import datetime
from decimal import Decimal

from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

# AgentCore imports
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

# ── Path setup ──────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
REPO_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "agents").is_dir() and (_parent / "mcp_servers").is_dir():
        REPO_ROOT = _parent
        break
sys.path.insert(0, str(REPO_ROOT))

MCP_SERVER_PATH = REPO_ROOT / "mcp_servers" / "socotra_mock"
MOCK_DATA_PATH = MCP_SERVER_PATH / "mock_data.json"

from agents import authenticator_agent
from agents import extractor_agent
from agents import policy_verification_agent

# ── AWS configuration ────────────────────────────────────────────────────────────
REGION = "us-east-1"
S3_BUCKET = os.environ["WORKSHOP_S3_BUCKET"]  # Set by lifecycle config or export manually
S3_PREFIX = "claims-processing/claimant-data/"
DYNAMODB_TABLE = "claims-metadata"

s3_client = boto3.client("s3", region_name=REGION)
dynamodb = boto3.resource("dynamodb", region_name=REGION)

print("✅ Setup complete")
print(f"   REPO_ROOT  : {REPO_ROOT}")
print(f"   Region     : {REGION}")

## Part A — The "Before": Stateless Pipeline

Let’s rebuild the Intake Orchestrator and run it. Then we’ll try to ask a follow-up question about the claim — and watch it fail.

This establishes the baseline: **without AgentCore, every pipeline run is isolated. No session state, no memory, no context for downstream phases.**

### Rebuild the Intake Orchestrator Tools

We define the same three `@tool` functions from previous notebooks. This is identical code — we’re establishing the baseline before adding AgentCore.

In [ ]:
# ── Tool 1: Retrieve claim documents from S3 ───────────────────────────────────

LOCAL_DOCS_DIR = Path("claim_documents")
LOCAL_DOCS_DIR.mkdir(exist_ok=True)

@tool
def retrieve_claim_documents(s3_bucket: str, s3_prefix: str) -> str:
    """Download claim PDF documents from S3.

    Args:
        s3_bucket: S3 bucket name containing claim documents
        s3_prefix: S3 key prefix for the claim's documents
    """
    response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)
    s3_objects = response.get("Contents", [])
    downloaded = []
    for obj in s3_objects:
        key = obj["Key"]
        filename = key.split("/")[-1]
        if not filename or not filename.endswith(".pdf"): continue
        local_path = LOCAL_DOCS_DIR / filename
        s3_client.download_file(s3_bucket, key, str(local_path))
        downloaded.append(str(local_path))
    return json.dumps({"status": "success", "documents_retrieved": len(downloaded), "document_paths": downloaded})

# ── Tool 2: Persist claim record to DynamoDB ──────────────────────────────────

@tool
def persist_claim_to_dynamodb(
    claim_id: str, policy_number: str, claimant_name: str,
    date_of_death: str, auth_summary: str, verification_summary: str,
) -> str:
    """Write the final claim record to DynamoDB.

    Args:
        claim_id: Unique claim identifier
        policy_number: Policy number
        claimant_name: Name of the claimant
        date_of_death: Date of death (YYYY-MM-DD)
        auth_summary: Summary of authentication result
        verification_summary: Summary of policy verification result
    """
    from botocore.exceptions import ClientError
    verification_decision = {"raw_response": verification_summary[:3000]}
    def float_to_decimal(obj):
        if isinstance(obj, float): return Decimal(str(obj))
        elif isinstance(obj, dict): return {k: float_to_decimal(v) for k, v in obj.items()}
        elif isinstance(obj, list): return [float_to_decimal(i) for i in obj]
        return obj
    claim_record = {
        "claim_id": claim_id, "policy_number": policy_number,
        "claimant_name": claimant_name, "date_of_death": date_of_death,
        "auth_result_summary": auth_summary[:2000],
        "verification_decision": verification_decision,
        "created_at": datetime.now(datetime.timezone.utc).isoformat() + "Z",
        "stage": "intake_orchestration_complete", "pipeline": "intake_orchestrator_v1",
    }
    try:
        table = dynamodb.create_table(TableName=DYNAMODB_TABLE,
            KeySchema=[{"AttributeName": "claim_id", "KeyType": "HASH"}],
            AttributeDefinitions=[{"AttributeName": "claim_id", "AttributeType": "S"}],
            BillingMode="PAY_PER_REQUEST")
        table.wait_until_exists()
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceInUseException":
            table = dynamodb.Table(DYNAMODB_TABLE)
        else: raise
    table.put_item(Item=float_to_decimal(claim_record))
    return json.dumps({"status": "success", "claim_id": claim_id})

# ── Tool 3: Run the pre-processing graph ─────────────────────────────────────

SOCOTRA_SERVER_SCRIPT = str(MCP_SERVER_PATH / "server.py")
socotra_mcp_client = MCPClient(
    lambda: stdio_client(StdioServerParameters(
        command=sys.executable, args=[SOCOTRA_SERVER_SCRIPT],
        env={"MOCK_DATA_PATH": str(MOCK_DATA_PATH), **os.environ}
    ))
)
auth_passed = True

@tool
def run_preprocessing_graph(claim_prompt: str) -> str:
    """Run the 3-node pre-processing graph: authenticate, extract, verify_policy.

    Args:
        claim_prompt: Full claim submission prompt with all claim details
    """
    global auth_passed
    auth_passed = True
    mcp_tools = socotra_mcp_client.list_tools_sync()
    auth_agent = authenticator_agent.build_agent(mcp_tools)
    output_dir = Path("extracted_output")
    output_dir.mkdir(exist_ok=True)
    extract_agent = extractor_agent.build_agent(output_dir=output_dir)
    verify_agent = policy_verification_agent.build_agent(mcp_tools)
    def check_auth_passed(state): return auth_passed
    builder = GraphBuilder()
    builder.add_node(auth_agent, "authenticate")
    builder.add_node(extract_agent, "extract")
    builder.add_node(verify_agent, "verify_policy")
    builder.set_entry_point("authenticate")
    builder.add_edge("authenticate", "extract", condition=check_auth_passed)
    builder.add_edge("extract", "verify_policy")
    graph = builder.build()
    return str(graph(claim_prompt))

print("✅ All three tools defined (same as NB 01_multi_agent_orchestration)")

### Run the Baseline Pipeline (No AgentCore)

We build the supervisor with no session manager — exactly as in NB 01_multi_agent_orchestration — and run the claim through the pipeline.

In [ ]:
# Build supervisor WITHOUT AgentCore (baseline)
SUPERVISOR_SYSTEM_PROMPT = """
You are the Intake Orchestrator — the supervisor agent for an Insurance Claims Processing worfklow.

Your Role:
You coordinate the full claim lifecycle. In this phase (Phase 1: Pre-Processing), you:
1. Retrieve claim documents from S3
2. Run the pre-processing graph (authenticate → extract → verify_policy)
3. Persist the claim record to DynamoDB

Your Tools:
- **retrieve_claim_documents(s3_bucket, s3_prefix)**: Download claim PDFs from S3
- **run_preprocessing_graph(claim_prompt)**: Execute the 3-node specialist agent graph
- **persist_claim_to_dynamodb(...)**: Write the final claim record to DynamoDB

##Workflow
When you receive a claim submission:
1. Call retrieve_claim_documents to download the claim documents from S3
2. Build a detailed prompt with all claim details and the EXACT document_paths returned by retrieve_claim_documents. Do NOT construct filenames yourself.
3. Call run_preprocessing_graph with that prompt
4. Review the graph output for authentication status and verification decision
5. Call persist_claim_to_dynamodb with the claim details and results
6. Summarize the outcome

## Guidelines
- Always retrieve documents before running the graph
- Always persist results, even if authentication failed — we need a record of rejected claims
- Include the claim_id, policy_number, claimant_name, and date_of_death when persisting
- For auth_summary, use the authentication portion of the graph output
- For verification_summary, use the policy verification portion of the graph output
- In your final summary, report what YOU completed (documents retrieved, graph executed, record persisted) 
— do NOT echo the specialist agents' next-step recommendations. The pre-processing phase is complete when you persist the record. The next phase (Adjudication) is handled separately.
"""

supervisor_model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0", region_name=REGION,
    additional_request_fields={"reasoningConfig": {"type": "enabled", "maxReasoningEffort": "low"}}
)

# No session_manager — this is the stateless baseline
intake_orchestrator = Agent(
    model=supervisor_model,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    tools=[retrieve_claim_documents, run_preprocessing_graph, persist_claim_to_dynamodb],
)

CLAIM = {
    "claim_id": "CLM-2026-00101", "policy_number": "WL-4582-1093",
    "claimant_name": "Lisa Doe", "date_of_death": "2026-01-15",
    "death_circumstances": "Natural causes — congestive heart failure at residence",
}

supervisor_prompt = (
    f"## New Claim\n"
    f"Claim ID: {CLAIM['claim_id']}\n"
    f"Policy: {CLAIM['policy_number']}\n"
    f"Claimant: {CLAIM['claimant_name']}\n"
    f"Date of Death: {CLAIM['date_of_death']}\n"
    f"Circumstances: {CLAIM['death_circumstances']}\n"
    f"S3 Bucket: {S3_BUCKET}\nS3 Prefix: {S3_PREFIX}\n"
    "Process this claim through the full pre-processing pipeline."
)

print("🚀 Running baseline pipeline (no AgentCore)...")
print("=" * 60)
with socotra_mcp_client:
    baseline_result = intake_orchestrator(supervisor_prompt)
print("=" * 60)
print("✅ Baseline pipeline complete")

### The Problem: No Memory

Now let’s simulate what happens when a downstream agent (or the same supervisor in a new session) tries to ask about the claim. We create a **fresh** agent instance — simulating a new session — and ask it about the claim we just processed.

In [ ]:
# Simulate a new session — fresh agent, no history
fresh_orchestrator = Agent(
    model=supervisor_model,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    tools=[retrieve_claim_documents, run_preprocessing_graph, persist_claim_to_dynamodb],
)

followup = "What was the verification decision for claim CLM-2026-00101? Was the policy active? What exclusions were checked?"

print("🔍 Asking follow-up question in a new session (no AgentCore)...")
print(f"   Question: {followup}")
print()
print("Response from fresh agent:")
followup_result = fresh_orchestrator(followup)
print()
print("─" * 60)
#print(str(followup_result)[:1000])
print()
print("⚠️  Notice: The agent has NO memory of the previous pipeline run.")
print("   It cannot answer questions about the claim without re-running the entire pipeline.")
print("   This is the problem AgentCore Memory solves.")

## Part B — AgentCore Runtime: Session Isolation

In production, the Intake Orchestrator needs to handle multiple claims concurrently. AgentCore Runtime solves this by giving each claim its own **microVM** — isolated compute, memory, and filesystem that are completely separated from other sessions.

### How AgentCore Runtime Sessions Work

```
Claim A (session-001)  ──→  microVM-A  ──→  isolated Python process
Claim B (session-002)  ──→  microVM-B  ──→  isolated Python process
```

Each session gets a unique `runtimeSessionId`. The Runtime provisions a dedicated execution environment, preserves context between invocations within the same session, and terminates the microVM when the session ends — sanitizing all memory to prevent cross-session contamination.

### What This Means for Claims Processing

| Without Runtime | With Runtime |
|---|---|
| Shared Python process | Isolated microVM per claim |
| Global `auth_passed` flag shared | Each claim has its own state |
| No session lifecycle management | Auto-cleanup after idle timeout |
| Manual scaling | Managed scaling per session |

In this workshop, we demonstrate the Runtime concepts by wrapping our agent with the `BedrockAgentCoreApp` entrypoint pattern. In a production deployment, you would use `agentcore deploy` to host the agent in a managed Runtime environment.

### The AgentCore Runtime Entrypoint Pattern

To deploy an agent to AgentCore Runtime, you wrap it with `BedrockAgentCoreApp` and define an `@app.entrypoint` function. The Runtime calls this function for each invocation, passing the payload and a context object that includes the `session_id`.

Here’s what the Intake Orchestrator looks like as a Runtime-ready agent:

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AgentCore Runtime entrypoint pattern
#
# In production, this file would be deployed via `agentcore deploy`.
# The Runtime provisions a microVM per session and calls invoke() for
# each request. Here we show the pattern for reference.
# ─────────────────────────────────────────────────────────────────────────────

from bedrock_agentcore.runtime import BedrockAgentCoreApp

# This is how the agent would be structured for Runtime deployment:
#
# app = BedrockAgentCoreApp()
#
# @app.entrypoint
# def invoke(payload, context):
#     session_id = getattr(context, "session_id", None)
#     # Each invocation gets the session_id from the Runtime
#     # The same session_id routes to the same microVM
#     agent = Agent(
#         model=supervisor_model,
#         system_prompt=SUPERVISOR_SYSTEM_PROMPT,
#         tools=[retrieve_claim_documents, run_preprocessing_graph, persist_claim_to_dynamodb],
#     )
#     result = agent(payload.get("prompt", ""))
#     return {"response": str(result)}

# For this workshop, we simulate session isolation by tracking session IDs manually.
# In production, AgentCore Runtime handles this automatically.

SESSION_CLAIM_A = f"claim-session-{uuid.uuid4()}"
SESSION_CLAIM_B = f"claim-session-{uuid.uuid4()}"

print("✅ AgentCore Runtime entrypoint pattern demonstrated")
print(f"   Session A: {SESSION_CLAIM_A}")
print(f"   Session B: {SESSION_CLAIM_B}")
print()
print("   In production, each session gets its own microVM.")
print("   No shared state, no cross-session contamination.")

## Part C — AgentCore Memory: Short-Term (Session History)

AgentCore Memory captures the full conversation — every tool call, every agent response — as **events** in short-term memory. This is tied to a session: within the same session, the agent can recall what happened earlier.

### How It Works

1. **Create a Memory resource** — a named container for storing conversation events
2. **Configure `AgentCoreMemorySessionManager`** — wires the memory to the Strands agent
3. **Run the pipeline** — every tool call and response is automatically captured
4. **Query within the session** — the agent can answer questions about what it just did

The key integration point is the `session_manager` parameter on the Strands `Agent`. When set, the agent automatically persists all messages to AgentCore Memory.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1: Create an AgentCore Memory resource (short-term only for now)
# ─────────────────────────────────────────────────────────────────────────────

memory_client = MemoryClient(region_name=REGION)

# Create a basic memory resource (short-term memory)
try:
    stm_memory = memory_client.create_memory(
        name="IntakeOrchestrator_STM",
        description="Short-term memory for Intake Orchestrator pipeline sessions"
    )
    MEMORY_ID = stm_memory.get("id")
except Exception as e:
    if "already exists" in str(e):
        print("   Memory already exists, reusing...")
        memories = memory_client.list_memories()
        for m in memories:
            if m.get("name") == "IntakeOrchestrator_STM":
                MEMORY_ID = m.get("id")
                break
    else:
        raise

import time
MEMORY_ID = stm_memory.get("id")
print(f"   Memory ID: {MEMORY_ID}")
print("   Waiting for memory to become ACTIVE...")
for _ in range(30):
    status = memory_client.get_memory(memory_id=MEMORY_ID).get('status', 'CREATING')
    if status == 'ACTIVE':
        break
    time.sleep(5)
print(f"✅ AgentCore Memory is {status}")

### Run the Pipeline WITH AgentCore Memory

Now we rebuild the supervisor with `session_manager=AgentCoreMemorySessionManager(...)`. Every tool call and response will be automatically captured as events in short-term memory.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 2: Configure memory for the supervisor's session
# ─────────────────────────────────────────────────────────────────────────────

ACTOR_ID = "intake-orchestrator"
SESSION_ID_PHASE1 = f"phase1-{CLAIM['claim_id']}-{uuid.uuid4()}"

memory_config = AgentCoreMemoryConfig(
    memory_id=MEMORY_ID,
    session_id=SESSION_ID_PHASE1,
    actor_id=ACTOR_ID,
)

session_manager = AgentCoreMemorySessionManager(
    agentcore_memory_config=memory_config,
    region_name=REGION,
)

# Rebuild supervisor WITH session_manager
intake_orchestrator_with_memory = Agent(
    model=supervisor_model,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    tools=[retrieve_claim_documents, run_preprocessing_graph, persist_claim_to_dynamodb],
    session_manager=session_manager,  # <── THIS IS THE KEY CHANGE
)

print("🚀 Running pipeline WITH AgentCore Memory...")
print(f"   Memory ID   : {MEMORY_ID}")
print(f"   Session ID  : {SESSION_ID_PHASE1}")
print(f"   Actor ID    : {ACTOR_ID}")
print("=" * 60)

with socotra_mcp_client:
    memory_result = intake_orchestrator_with_memory(supervisor_prompt)

print("=" * 60)
print("✅ Pipeline complete — conversation captured in AgentCore Memory")

### Query Short-Term Memory (Same Session)

Now let’s verify the conversation was captured. We list the events stored in short-term memory for this session. Each event represents a turn in the conversation — user messages, tool calls, and agent responses.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Verify: list events in short-term memory for this session
# ─────────────────────────────────────────────────────────────────────────────

events = memory_client.list_events(
    memory_id=MEMORY_ID,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID_PHASE1,
)

event_list = events if isinstance(events, list) else events.get("events", [])
print(f"📋 Short-term memory events for session {SESSION_ID_PHASE1[:30]}...")
print(f"   Total events: {len(event_list)}")
print()
for i, evt in enumerate(event_list[:10]):
    messages = evt.get("messages", [])
    for msg in messages:
        role = msg.get("role", "unknown")
        content = msg.get("content", "")[:100]
        print(f"   Event {i}: [{role}] {content}...")
print()
print("✅ Conversation history is persisted in AgentCore Memory")
print("   This is short-term memory — available within the same session.")

## Part D — AgentCore Memory: Long-Term (Cross-Session Facts)

Short-term memory captures the raw conversation within a session. But what about **cross-session** context? When the Adjudicator starts a new session for the same claim, it needs the verification decision, not the raw tool call logs.

This is where **long-term memory strategies** come in. AgentCore automatically extracts structured insights from the raw conversation:

| Strategy | What It Extracts | Use Case |
|---|---|---|
| `summaryMemoryStrategy` | Session summaries | Quick overview of what happened in Phase 1 |
| `semanticMemoryStrategy` | Facts and knowledge | Policy number, verification status, claimant details |

Long-term memory extraction is **asynchronous** — it runs in the background after the conversation events are stored. Once extracted, these memories are retrievable from any session using `retrieve_memories()`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Create a comprehensive memory with long-term strategies
# ─────────────────────────────────────────────────────────────────────────────

ltm_memory = memory_client.create_memory_and_wait(
    name="IntakeOrchestrator_LTM",
    description="Long-term memory for claims processing pipeline",
    strategies=[
        {
            "summaryMemoryStrategy": {
                "name": "ClaimSessionSummarizer",
                "namespaces": ["/summaries/{actorId}/{sessionId}/"]
            }
        },
        {
            "semanticMemoryStrategy": {
                "name": "ClaimFactExtractor",
                "namespaces": ["/facts/{actorId}/"]
            }
        },
    ]
)

LTM_MEMORY_ID = ltm_memory.get("id")
print(f"✅ Long-term memory created")
print(f"   Memory ID : {LTM_MEMORY_ID}")
print(f"   Strategies: summaryMemoryStrategy, semanticMemoryStrategy")

### Run the Pipeline with Long-Term Memory

We run the pipeline again, this time with the long-term memory resource. After the pipeline completes, AgentCore will asynchronously extract summaries and facts from the conversation.

In [ ]:
# Configure memory with LTM strategies
SESSION_ID_LTM = f"phase1-ltm-{CLAIM['claim_id']}-{uuid.uuid4()}"

ltm_config = AgentCoreMemoryConfig(
    memory_id=LTM_MEMORY_ID,
    session_id=SESSION_ID_LTM,
    actor_id=ACTOR_ID,
)

ltm_session_manager = AgentCoreMemorySessionManager(
    agentcore_memory_config=ltm_config,
    region_name=REGION,
)

# Rebuild supervisor with LTM session manager
intake_orchestrator_ltm = Agent(
    model=supervisor_model,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    tools=[retrieve_claim_documents, run_preprocessing_graph, persist_claim_to_dynamodb],
    session_manager=ltm_session_manager,
)

print("🚀 Running pipeline with long-term memory...")
print(f"   LTM Memory ID : {LTM_MEMORY_ID}")
print(f"   Session ID    : {SESSION_ID_LTM}")
print("=" * 60)

with socotra_mcp_client:
    ltm_result = intake_orchestrator_ltm(supervisor_prompt)

print("=" * 60)
print("✅ Pipeline complete — long-term memory extraction will run in background")
print("   Waiting 30 seconds for extraction to complete...")
time.sleep(30)
print("✅ Extraction should be complete")

## Part E — The Payoff: Cross-Session Context Retrieval

This is the "after" moment. We start a **completely new session** — simulating the Adjudicator agent in Phase 2 — and retrieve the Phase 1 context from AgentCore Memory without re-running the pipeline.

### What We’re Proving

| Before (Part A) | After (Part E) |
|---|---|
| Fresh agent has no memory | Fresh agent retrieves facts from LTM |
| Must re-run pipeline to get verification decision | Verification decision available instantly |
| No session summary | Session summary available for audit trail |
| Downstream agents are blind | Downstream agents have full Phase 1 context |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Retrieve session summary from long-term memory
# ─────────────────────────────────────────────────────────────────────────────

# Query 1: Get the session summary
summary_namespace = f"/summaries/{ACTOR_ID}/{SESSION_ID_LTM}/"
summaries = memory_client.retrieve_memories(
    memory_id=LTM_MEMORY_ID,
    namespace=summary_namespace,
    query="summarize the claim processing session"
)

print("📋 Session Summary (from long-term memory)")
print("=" * 60)
summary_records = summaries if isinstance(summaries, list) else summaries.get("memoryRecords", [])
if summary_records:
    for rec in summary_records:
        print(rec.get("content", {}).get("text", "No text")[:500])
        print()
else:
    print("   No summaries found yet. Extraction may still be in progress.")
    print("   Try re-running this cell after 30 more seconds.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Query 2: Retrieve extracted facts (semantic memory)
# This is what the Adjudicator would use in Phase 2
# ─────────────────────────────────────────────────────────────────────────────

facts_namespace = f"/facts/{ACTOR_ID}/"

# Query for verification decision
verification_facts = memory_client.retrieve_memories(
    memory_id=LTM_MEMORY_ID,
    namespace=facts_namespace,
    query="What was the policy verification decision? Was the policy active? Were any exclusions triggered?"
)

print("🔍 Semantic Facts: Verification Decision")
print("=" * 60)
fact_records = verification_facts if isinstance(verification_facts, list) else verification_facts.get("memoryRecords", [])
if fact_records:
    for rec in fact_records[:5]:
        print(rec.get("content", {}).get("text", "No text")[:500])
        print()
else:
    print("   No facts extracted yet. Try re-running after 30 more seconds.")

# Query for claimant details
claimant_facts = memory_client.retrieve_memories(
    memory_id=LTM_MEMORY_ID,
    namespace=facts_namespace,
    query="Who is the claimant? What is the policy number and claim ID?"
)

print()
print("🔍 Semantic Facts: Claimant Details")
print("=" * 60)
claimant_records = claimant_facts if isinstance(claimant_facts, list) else claimant_facts.get("memoryRecords", [])
if claimant_records:
    for rec in claimant_records[:5]:
        print(rec.get("content", {}).get("text", "No text")[:500])
        print()
else:
    print("   No facts extracted yet.")

### Simulate the Adjudicator Retrieving Phase 1 Context

This is the key payoff. We create a **new agent in a new session** — simulating the Adjudicator starting Phase 2 — and use `retrieve_memories()` to pull the verification decision and session summary. The Adjudicator gets full Phase 1 context without re-running the pre-processing pipeline.

Compare this to Part A, where a fresh agent had zero knowledge of the previous run.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Simulate the Adjudicator (Phase 2) retrieving Phase 1 context
# ─────────────────────────────────────────────────────────────────────────────

# New session for the Adjudicator
ADJUDICATOR_SESSION = f"phase2-adjudicator-{uuid.uuid4()}"

# Configure memory retrieval for the new session
adjudicator_config = AgentCoreMemoryConfig(
    memory_id=LTM_MEMORY_ID,
    session_id=ADJUDICATOR_SESSION,
    actor_id="adjudicator",
    retrieval_config={
        f"/facts/{ACTOR_ID}/": RetrievalConfig(top_k=5, relevance_score=0.5),
        f"/summaries/{ACTOR_ID}/{SESSION_ID_LTM}/": RetrievalConfig(top_k=3, relevance_score=0.5),
    }
)

adjudicator_session_mgr = AgentCoreMemorySessionManager(
    agentcore_memory_config=adjudicator_config,
    region_name=REGION,
)

# Create a simple Adjudicator agent that uses the retrieved context
# Note this is just to test the output retrieved from AgentCore Memory
# Adjudicator workflow is handled in Human in the Loop Integration notebook
adjudicator_agent = Agent(
    model=supervisor_model,
    system_prompt="You are the Adjudicator for Insurance Claims. You review pre-processing results and make claim decisions. Use the context from memory to answer questions about claims.",
    session_manager=adjudicator_session_mgr,
)

# Ask the SAME follow-up question from Part A
adjudicator_query = "What was the verification decision for claim CLM-2026-00101? Was the policy active? What exclusions were checked?"

print("🔍 Adjudicator asking about Phase 1 results (NEW session, WITH AgentCore Memory)...")
print(f"   Query: {adjudicator_query}")
print()
adjudicator_result = adjudicator_agent(adjudicator_query)
print()
print("─" * 60)
print("Adjudicator response:")
print(str(adjudicator_result)[:2000])
print()
print("✅ The Adjudicator has full Phase 1 context WITHOUT re-running the pipeline.")
print("   Compare this to Part A where the fresh agent had zero knowledge.")

## Part F — Cleanup

Delete the AgentCore Memory resources created during this notebook.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cleanup: delete memory resources
# ─────────────────────────────────────────────────────────────────────────────

try:
    memory_client.delete_memory(memory_id=MEMORY_ID)
    print(f"✅ Deleted STM memory: {MEMORY_ID}")
except Exception as e:
    print(f"⚠️  Could not delete STM memory: {e}")

try:
    memory_client.delete_memory(memory_id=LTM_MEMORY_ID)
    print(f"✅ Deleted LTM memory: {LTM_MEMORY_ID}")
except Exception as e:
    print(f"⚠️  Could not delete LTM memory: {e}")

import shutil
for folder in ['claim_documents', 'extracted_output']:
    if Path(folder).exists():
        shutil.rmtree(folder)
        print(f'✅ Removed {folder}/')
    else:
        print(f'ℹ️  {folder}/ not found (already clean)')

## ✅ Notebook Complete: AgentCore Integration

### What You Built

| Component | What It Does |
|---|---|
| **AgentCore Runtime** | Session isolation via microVMs — each claim gets its own execution environment with no shared state |
| **AgentCore Memory (STM)** | Short-term memory captures the full pipeline conversation within a session |
| **AgentCore Memory (LTM)** | Long-term memory extracts facts and summaries that persist across sessions |
| **Cross-session retrieval** | Downstream agents (Adjudicator) retrieve Phase 1 context without re-running the pipeline |

### Before vs After

| | Before (notebook 03) | After (notebook 04) |
|---|---|---|
| **Session isolation** | Shared kernel process | Isolated microVM per claim |
| **Within-session memory** | Agent message history (lost on restart) | Persisted in AgentCore STM |
| **Cross-session context** | None — must re-run pipeline | Semantic facts + summaries in LTM |
| **Downstream agent access** | Blind — no Phase 1 context | Full context via `retrieve_memories()` |

### What’s Next

| Phase | What Gets Added |
|---|---|
| Claims Processing | Adjudicator agent with human-in-the-loop, retrieves Phase 1 context from AgentCore Memory |